<a href="https://colab.research.google.com/github/lamonega/colab-llm/blob/main/colab_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import time
import requests
import threading
import os

# ------------------------------------------------------------
# Configuración inicial
# ------------------------------------------------------------
MODEL_NAME = "qwen3:14b"

# Fijar variables de entorno que usa Ollama
os.environ['OLLAMA_CONTEXT_LENGTH'] = '16384'
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['MODEL_NAME'] = MODEL_NAME

# ------------------------------------------------------------
# Paso 1: instalar dependencias del sistema si no existen
# ------------------------------------------------------------
print("Verificando dependencias del sistema...")
subprocess.run(['apt-get', 'update'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', 'lshw', 'pciutils', 'zstd'], capture_output=True)

# Mostrar info de CUDA y GPU (opcional, pero útil)
subprocess.run(['nvcc', '--version'])
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'])

# ------------------------------------------------------------
# Paso 2: instalar Ollama si no está instalado
# ------------------------------------------------------------
print("Verificando si Ollama está instalado...")
result = subprocess.run(['which', 'ollama'], capture_output=True)
if result.returncode != 0:
    print("Ollama no está instalado. Instalando...")
    subprocess.run(['curl', '-fsSL', 'https://ollama.com/install.sh', '|', 'sh'], shell=True)
else:
    print("Ollama ya está instalado.")

# ------------------------------------------------------------
# Paso 3: iniciar el servicio de Ollama si no está corriendo
# ------------------------------------------------------------
def ollama_responds():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        return r.status_code == 200
    except:
        return False

if not ollama_responds():
    print("Iniciando servicio de Ollama...")
    def serve():
        subprocess.call(['ollama', 'serve'])
    hilo = threading.Thread(target=serve, daemon=True)
    hilo.start()
    # Esperar hasta que responda (hasta 90 segundos)
    for i in range(90):
        if ollama_responds():
            print(f"Servicio listo después de {i+1} segundos.")
            break
        time.sleep(1)
    else:
        raise RuntimeError("El servicio de Ollama no arrancó a tiempo.")
else:
    print("El servicio de Ollama ya estaba corriendo.")

# ------------------------------------------------------------
# Paso 4: descargar el modelo solo si no está en la lista local
# ------------------------------------------------------------
print("Verificando si el modelo ya está descargado...")
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
if MODEL_NAME in result.stdout:
    print(f"El modelo {MODEL_NAME} ya existe localmente.")
else:
    print(f"Descargando modelo {MODEL_NAME} (puede tardar varios minutos)...")
    pull = subprocess.run(['ollama', 'pull', MODEL_NAME], capture_output=True, text=True)
    if pull.returncode != 0:
        print(pull.stderr)
        raise RuntimeError(f"Error al descargar el modelo {MODEL_NAME}.")
    print("Modelo descargado correctamente.")

# ------------------------------------------------------------
# Paso 5: forzar la carga del modelo a VRAM
# ------------------------------------------------------------
print("Cargando modelo a VRAM...")
payload = {
    "model": MODEL_NAME,
    "prompt": "Hola",
    "stream": False
}
try:
    respuesta = requests.post("http://localhost:11434/api/generate", json=payload, timeout=120)
    if respuesta.status_code == 200:
        print("Modelo cargado en VRAM correctamente.")
    else:
        print(f"Atención: la respuesta tuvo código {respuesta.status_code}")
except Exception as e:
    raise RuntimeError(f"No se pudo cargar el modelo: {e}")

# ------------------------------------------------------------
# Todo listo
# ------------------------------------------------------------
print("Entorno local preparado. Podés continuar con la Celda 2.")

In [ ]:
import subprocess
import re
import time
import requests
import os

# ------------------------------------------------------------
# Obtener el nombre del modelo desde las variables de entorno
# ------------------------------------------------------------
model = os.environ.get('MODEL_NAME')
if not model:
    raise RuntimeError("No se encontró la variable MODEL_NAME. Ejecuta la Celda 1 primero.")

# ------------------------------------------------------------
# Paso 1: descargar cloudflared si no existe
# ------------------------------------------------------------
print("Verificando cloudflared...")
if not os.path.exists('./cloudflared'):
    print("Descargando cloudflared...")
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', 'cloudflared'])
    subprocess.run(['chmod', '+x', 'cloudflared'])
    print("cloudflared listo.")
else:
    print("cloudflared ya existe.")

# ------------------------------------------------------------
# Paso 2: iniciar el túnel y obtener la URL pública
# ------------------------------------------------------------
print("Iniciando túnel de Cloudflare...")
proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

public_url = None
inicio = time.time()

# Leer la salida del túnel línea por línea hasta encontrar la URL
for linea in proc.stdout:
    if "https://" in linea and "trycloudflare.com" in linea:
        # Extraer la URL con una expresión regular simple
        coincidencia = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', linea)
        if coincidencia:
            public_url = coincidencia.group(1)
            break
    # Si pasan 60 segundos, abortar
    if time.time() - inicio > 60:
        proc.terminate()
        raise RuntimeError("No se obtuvo la URL pública en 60 segundos.")

if not public_url:
    proc.terminate()
    raise RuntimeError("No se pudo extraer la URL pública.")

print("Túnel creado. URL obtenida.")

# ------------------------------------------------------------
# Paso 3: esperar que la URL pública sea accesible
# ------------------------------------------------------------
print("Verificando que la URL pública responda...")
for intento in range(1, 13):  # hasta 12 intentos (aprox 60 segundos)
    try:
        respuesta = requests.get(f"{public_url}/api/tags", timeout=5)
        if respuesta.status_code == 200:
            print(f"URL pública accesible (intento {intento}).")
            break
    except:
        pass
    time.sleep(5)
else:
    proc.terminate()
    raise RuntimeError("La URL pública no responde después de varios intentos.")

# ------------------------------------------------------------
# Paso 4: hacer una prueba sencilla al modelo
# ------------------------------------------------------------
print("Enviando pregunta de prueba al modelo...")
payload = {
    "model": model,
    "prompt": "¿Cuál es el resultado de sumar 2 + 2?",
    "stream": False
}

try:
    r = requests.post(f"{public_url}/api/generate", json=payload, timeout=120)
    r.raise_for_status()  # Lanza excepción si el código HTTP no es 200
    datos = r.json()
    if datos.get('response'):
        print("Modelo responde correctamente.")
    else:
        raise RuntimeError("La respuesta del modelo está vacía.")
except Exception as e:
    proc.terminate()
    raise RuntimeError(f"Error al consultar el modelo: {e}")

# ------------------------------------------------------------
# Paso 5: mostrar la URL pública
# ------------------------------------------------------------
print("------------------------------------------------------------")
print("TODO LISTO. El servicio está activo y accesible desde internet.")
print(f"URL pública: {public_url}")
print("------------------------------------------------------------")